# XGBoost con Alpha Vantage

En este notebook voy a entrenar un modelo XGBoost para comprobar si puede encontrar relaciones no lineales más estables que Random Forest

Primero utilizaré una configuración base. Después ajustaré sus parámetros mediante divisiones temporales dentro del entrenamiento. La validación principal se utilizará únicamente después del ajuste y la prueba final continuará separada.

In [1]:
from pathlib import Path

import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    TimeSeriesSplit
)
from xgboost import XGBClassifier

print("Versión de XGBoost:", xgb.__version__)

# Localizo la carpeta principal
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual

archivo_particiones = (
    ruta_proyecto
    / "data"
    / "processed"
    / "eurusd_alpha_vantage_particiones.csv"
)

datos = pd.read_csv(
    archivo_particiones,
    parse_dates=["Date"],
    index_col="Date"
)

datos = datos.sort_index()
datos["Objetivo"] = datos["Objetivo"].astype("Int64")

print("Dimensiones:", datos.shape)

Versión de XGBoost: 3.3.0
Dimensiones: (4980, 21)


In [2]:
variables_predictoras = [
    "Retorno_diario",
    "Retorno_lag_1",
    "Retorno_lag_2",
    "Retorno_lag_3",
    "Retorno_lag_5",
    "Rango_diario",
    "Cuerpo_vela",
    "Posicion_cierre",
    "Distancia_MA5",
    "Distancia_MA10",
    "Distancia_MA20",
    "Volatilidad_5",
    "Volatilidad_20",
    "RSI_14",
    "MACD_hist"
]

# Utilizo únicamente entrenamiento y validación
entrenamiento = datos[
    datos["Particion"] == "entrenamiento"
].copy()

validacion = datos[
    datos["Particion"] == "validacion"
].copy()

X_entrenamiento = entrenamiento[
    variables_predictoras
]

y_entrenamiento = entrenamiento[
    "Objetivo"
].astype(int)

X_validacion = validacion[
    variables_predictoras
]

y_validacion = validacion[
    "Objetivo"
].astype(int)

print("Entrenamiento:", X_entrenamiento.shape)
print("Validación:", X_validacion.shape)

Entrenamiento: (4052, 15)
Validación: (522, 15)


In [3]:
# Utilizo la misma función para comparar las configuraciones
def calcular_metricas(
    nombre,
    particion,
    y_real,
    y_predicho,
    probabilidades
):
    return {
        "Modelo": nombre,
        "Particion": particion,
        "Accuracy": accuracy_score(
            y_real,
            y_predicho
        ),
        "Balanced_accuracy": balanced_accuracy_score(
            y_real,
            y_predicho
        ),
        "Precision": precision_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "Recall": recall_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "F1": f1_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_real,
            probabilidades
        )
    }


columnas_metricas = [
    "Accuracy",
    "Balanced_accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

resultados_xgb = []

# Configuración base

Primero voy a entrenar una versión base de XGBoost. Esta configuración servirá como referencia para comprobar si el ajuste temporal mejora realmente el comportamiento del modelo.

También compararé entrenamiento y validación para detectar posibles señales de sobreajuste.

In [4]:
modelo_xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    importance_type="gain",
    random_state=42,
    n_jobs=-1
)

modelo_xgb_base.fit(
    X_entrenamiento,
    y_entrenamiento
)

pred_xgb_base_entrenamiento = (
    modelo_xgb_base.predict(
        X_entrenamiento
    )
)

pred_xgb_base_validacion = (
    modelo_xgb_base.predict(
        X_validacion
    )
)

prob_xgb_base_entrenamiento = (
    modelo_xgb_base.predict_proba(
        X_entrenamiento
    )[:, 1]
)

prob_xgb_base_validacion = (
    modelo_xgb_base.predict_proba(
        X_validacion
    )[:, 1]
)

resultados_xgb.append(
    calcular_metricas(
        "XGBoost base",
        "Entrenamiento",
        y_entrenamiento,
        pred_xgb_base_entrenamiento,
        prob_xgb_base_entrenamiento
    )
)

resultados_xgb.append(
    calcular_metricas(
        "XGBoost base",
        "Validación",
        y_validacion,
        pred_xgb_base_validacion,
        prob_xgb_base_validacion
    )
)

# Ajuste de parámetros con validación temporal

Voy a probar distintas configuraciones utilizando únicamente el conjunto de entrenamiento.

Las divisiones internas respetarán el orden cronológico y dejarán una fila de separación entre entrenamiento y validación interna, porque el objetivo de cada fila depende de la jornada siguiente.

La búsqueda evaluará principalmente la balanced accuracy, pero también guardará el ROC-AUC para revisar la calidad de las probabilidades.

In [5]:
# Creo divisiones internas respetando el orden temporal
validacion_temporal = TimeSeriesSplit(
    n_splits=5,
    gap=1
)

# Dejo un solo hilo dentro de XGBoost porque la búsqueda ya trabaja en paralelo
modelo_xgb_busqueda = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    importance_type="gain",
    random_state=42,
    n_jobs=1
)

parametros_xgb = {
    "n_estimators": [
        100,
        200,
        400,
        800
    ],
    "learning_rate": [
        0.01,
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        2,
        3,
        4,
        5
    ],
    "min_child_weight": [
        1,
        5,
        10,
        20
    ],
    "subsample": [
        0.6,
        0.8,
        1.0
    ],
    "colsample_bytree": [
        0.6,
        0.8,
        1.0
    ],
    "gamma": [
        0,
        0.05,
        0.10,
        0.25
    ],
    "reg_alpha": [
        0,
        0.01,
        0.10,
        1.0
    ],
    "reg_lambda": [
        1,
        5,
        10,
        20
    ]
}

metricas_busqueda = {
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc"
}

busqueda_xgb = RandomizedSearchCV(
    estimator=modelo_xgb_busqueda,
    param_distributions=parametros_xgb,
    n_iter=40,
    scoring=metricas_busqueda,
    refit="balanced_accuracy",
    cv=validacion_temporal,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    error_score="raise",
    return_train_score=True
)

busqueda_xgb.fit(
    X_entrenamiento,
    y_entrenamiento
)

modelo_xgb_ajustado = (
    busqueda_xgb.best_estimator_
)

mejor_indice = (
    busqueda_xgb.best_index_
)

mejor_roc_auc_interno = (
    busqueda_xgb
    .cv_results_["mean_test_roc_auc"][
        mejor_indice
    ]
)

print("Mejor balanced accuracy interna:")
print(
    round(
        busqueda_xgb.best_score_,
        4
    )
)

print("\nROC-AUC interno de esa configuración:")
print(
    round(
        mejor_roc_auc_interno,
        4
    )
)

print("\nMejores parámetros:")
print(
    busqueda_xgb.best_params_
)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Mejor balanced accuracy interna:
0.5217

ROC-AUC interno de esa configuración:
0.5135

Mejores parámetros:
{'subsample': 1.0, 'reg_lambda': 20, 'reg_alpha': 0.01, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 2, 'learning_rate': 0.1, 'gamma': 0.05, 'colsample_bytree': 1.0}


In [6]:
pred_xgb_ajustado_entrenamiento = (
    modelo_xgb_ajustado.predict(
        X_entrenamiento
    )
)

pred_xgb_ajustado_validacion = (
    modelo_xgb_ajustado.predict(
        X_validacion
    )
)

prob_xgb_ajustado_entrenamiento = (
    modelo_xgb_ajustado.predict_proba(
        X_entrenamiento
    )[:, 1]
)

prob_xgb_ajustado_validacion = (
    modelo_xgb_ajustado.predict_proba(
        X_validacion
    )[:, 1]
)

resultados_xgb.append(
    calcular_metricas(
        "XGBoost ajustado",
        "Entrenamiento",
        y_entrenamiento,
        pred_xgb_ajustado_entrenamiento,
        prob_xgb_ajustado_entrenamiento
    )
)

resultados_xgb.append(
    calcular_metricas(
        "XGBoost ajustado",
        "Validación",
        y_validacion,
        pred_xgb_ajustado_validacion,
        prob_xgb_ajustado_validacion
    )
)

tabla_resultados_xgb = pd.DataFrame(
    resultados_xgb
)

tabla_resultados_xgb[columnas_metricas] = (
    tabla_resultados_xgb[columnas_metricas]
    .round(4)
)

display(
    tabla_resultados_xgb.set_index(
        ["Modelo", "Particion"]
    )
)

Accuracy  Balanced_accuracy  Precision  \
Modelo           Particion                                               
XGBoost base     Entrenamiento    0.9933             0.9933     0.9922   
                 Validación       0.4713             0.4722     0.4653   
XGBoost ajustado Entrenamiento    0.6412             0.6410     0.6383   
                 Validación       0.4521             0.4526     0.4457   

                                Recall      F1  ROC_AUC  
Modelo           Particion                               
XGBoost base     Entrenamiento  0.9946  0.9934   0.9999  
                 Validación     0.5234  0.4926   0.4616  
XGBoost ajustado Entrenamiento  0.6652  0.6515   0.6976  
                 Validación     0.4805  0.4624   0.4540

# Estabilidad anual en validación

Voy a evaluar el modelo ajustado por separado en 2023 y 2024. Esto permitirá comprobar si el resultado general se mantiene durante ambos años o si vuelve a depender de un único periodo

In [7]:
# Identifico el año de la jornada que se intenta predecir
fecha_objetivo = (
    datos.index
    .to_series()
    .shift(-1)
)

anio_objetivo_validacion = (
    fecha_objetivo
    .loc[validacion.index]
    .dt.year
)

pred_xgb_validacion_serie = pd.Series(
    pred_xgb_ajustado_validacion,
    index=validacion.index
)

prob_xgb_validacion_serie = pd.Series(
    prob_xgb_ajustado_validacion,
    index=validacion.index
)

resultados_anuales_xgb = []

for anio in [2023, 2024]:
    mascara_anio = (
        anio_objetivo_validacion == anio
    )

    resultado_anio = calcular_metricas(
        "XGBoost ajustado",
        str(anio),
        y_validacion.loc[
            mascara_anio
        ],
        pred_xgb_validacion_serie.loc[
            mascara_anio
        ],
        prob_xgb_validacion_serie.loc[
            mascara_anio
        ]
    )

    resultado_anio["Filas"] = (
        mascara_anio.sum()
    )

    resultados_anuales_xgb.append(
        resultado_anio
    )

tabla_anual_xgb = pd.DataFrame(
    resultados_anuales_xgb
)

tabla_anual_xgb = tabla_anual_xgb[
    [
        "Modelo",
        "Particion",
        "Filas",
        "Accuracy",
        "Balanced_accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ]
]

tabla_anual_xgb[columnas_metricas] = (
    tabla_anual_xgb[columnas_metricas]
    .round(4)
)

display(
    tabla_anual_xgb.set_index(
        ["Modelo", "Particion"]
    )
)

Filas  Accuracy  Balanced_accuracy  Precision  \
Modelo           Particion                                                  
XGBoost ajustado 2023         260    0.4500             0.4497     0.4593   
                 2024         262    0.4542             0.4561     0.4326   

                            Recall      F1  ROC_AUC  
Modelo           Particion                           
XGBoost ajustado 2023       0.4697  0.4644   0.4348  
                 2024       0.4919  0.4604   0.4732

# Importancia de las variables

Voy a revisar qué variables aportaron más ganancia al modelo.

La importancia no representa causalidad, pero permite comprobar si XGBoost depende excesivamente de una sola variable, especialmente de Posicion_cierre

In [8]:
importancia_xgb = pd.DataFrame({
    "Variable": variables_predictoras,
    "Importancia": (
        modelo_xgb_ajustado
        .feature_importances_
    )
})

importancia_xgb = (
    importancia_xgb
    .sort_values(
        "Importancia",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    importancia_xgb.round(4)
)

,Variable,Importancia
0,Retorno_diario,0.0848
1,MACD_hist,0.0749
2,Posicion_cierre,0.0746
3,Retorno_lag_1,0.0723
4,Distancia_MA5,0.0717
5,Cuerpo_vela,0.0688
6,Retorno_lag_3,0.0682
7,Distancia_MA10,0.0679
8,Rango_diario,0.0662
9,Retorno_lag_5,0.0623


# Comparación sin Posicion_cierre

Voy a repetir el modelo ajustado sin Posicion_cierre, utilizando los mismos parámetros seleccionados.

Esta comparación no busca realizar una segunda optimización, sino comprobar si el rendimiento cambia de forma importante al retirar la variable que produjo el resultado sospechoso con Yahoo Finance.

In [9]:
variables_sin_posicion = [
    variable
    for variable in variables_predictoras
    if variable != "Posicion_cierre"
]

X_entrenamiento_sin_posicion = entrenamiento[
    variables_sin_posicion
]

X_validacion_sin_posicion = validacion[
    variables_sin_posicion
]

# Mantengo los mismos parámetros para hacer una comparación directa
modelo_xgb_sin_posicion = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    importance_type="gain",
    random_state=42,
    n_jobs=-1,
    **busqueda_xgb.best_params_
)

modelo_xgb_sin_posicion.fit(
    X_entrenamiento_sin_posicion,
    y_entrenamiento
)

pred_xgb_sin_posicion_entrenamiento = (
    modelo_xgb_sin_posicion.predict(
        X_entrenamiento_sin_posicion
    )
)

pred_xgb_sin_posicion_validacion = (
    modelo_xgb_sin_posicion.predict(
        X_validacion_sin_posicion
    )
)

prob_xgb_sin_posicion_entrenamiento = (
    modelo_xgb_sin_posicion.predict_proba(
        X_entrenamiento_sin_posicion
    )[:, 1]
)

prob_xgb_sin_posicion_validacion = (
    modelo_xgb_sin_posicion.predict_proba(
        X_validacion_sin_posicion
    )[:, 1]
)

comparacion_posicion_xgb = [
    calcular_metricas(
        "XGBoost ajustado",
        "Entrenamiento",
        y_entrenamiento,
        pred_xgb_ajustado_entrenamiento,
        prob_xgb_ajustado_entrenamiento
    ),
    calcular_metricas(
        "XGBoost ajustado",
        "Validación",
        y_validacion,
        pred_xgb_ajustado_validacion,
        prob_xgb_ajustado_validacion
    ),
    calcular_metricas(
        "XGBoost sin Posicion_cierre",
        "Entrenamiento",
        y_entrenamiento,
        pred_xgb_sin_posicion_entrenamiento,
        prob_xgb_sin_posicion_entrenamiento
    ),
    calcular_metricas(
        "XGBoost sin Posicion_cierre",
        "Validación",
        y_validacion,
        pred_xgb_sin_posicion_validacion,
        prob_xgb_sin_posicion_validacion
    )
]

tabla_posicion_xgb = pd.DataFrame(
    comparacion_posicion_xgb
)

tabla_posicion_xgb[columnas_metricas] = (
    tabla_posicion_xgb[columnas_metricas]
    .round(4)
)

display(
    tabla_posicion_xgb.set_index(
        ["Modelo", "Particion"]
    )
)

Accuracy  Balanced_accuracy  \
Modelo                      Particion                                    
XGBoost ajustado            Entrenamiento    0.6412             0.6410   
                            Validación       0.4521             0.4526   
XGBoost sin Posicion_cierre Entrenamiento    0.6417             0.6414   
                            Validación       0.4464             0.4473   

                                           Precision  Recall      F1  ROC_AUC  
Modelo                      Particion                                          
XGBoost ajustado            Entrenamiento     0.6383  0.6652  0.6515   0.6976  
                            Validación        0.4457  0.4805  0.4624   0.4540  
XGBoost sin Posicion_cierre Entrenamiento     0.6370  0.6725  0.6543   0.6970  
                            Validación        0.4425  0.4961  0.4678   0.4533